<a href="https://colab.research.google.com/github/Piyush-Sharma-1/Diffusion_from_Scratch/blob/main/Implement_VAE_LOSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Maths to Magic

## Week 2 Assignment: Implement and Verify the VAE Loss

### Submitted By:

**Piyush Sharma**  
**Roll Number: 24B0310**

---

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence


## Part 1 — Closed-Form KL Divergence

In [ ]:
def kl_divergence_gaussian(mu: torch.Tensor,
                           logvar: torch.Tensor) -> torch.Tensor:
    """
    Closed-form KL divergence between
    N(mu, exp(logvar)) and N(0,1)
    Returns a scalar.
    """

    kl = -0.5 * torch.sum(
        1 + logvar - mu**2 - torch.exp(logvar)
    )

    return kl

In [ ]:
# Test it
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

Your KL:   458.7389
Torch KL:  458.7389
✅ Part 1 passed


## Part 2 — Reparameterization Trick

In [ ]:
def reparameterize(mu: torch.Tensor,
                   logvar: torch.Tensor) -> torch.Tensor:
    """
    Sample z using the reparameterization trick.
    z = mu + std * eps, where eps ~ N(0, I)
    """

    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps

    return z


In [ ]:
# Verify: the mean of many samples should be close to mu
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
# Means and stds should be close (within ~0.05)
print("✅ Part 2 passed")

Target mu:       [2.0, -1.0]
Sample mean:     [1.992883563041687, -0.9819331169128418]
Target std:      [1.0, 1.2840254306793213]
Sample std:      [0.9979977011680603, 1.2786657810211182]
✅ Part 2 passed


## Part 3 — ELBO Loss

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor
              ) -> torch.Tensor:

    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction="sum"
    )

    kl = kl_divergence_gaussian(
        mu,
        logvar
    )

    return recon_loss + kl


In [ ]:

# Quick smoke test
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")


ELBO loss (should be a positive scalar): 10592.2900
✅ Part 3 passed


## Part 4 — Reflection Answers

### Q1. Why can't we maximize log p(x) directly?

The direct maximization of log p(x) is not possible as it involves integration over all the latent variables which could generate the data set. However, in some complex cases, it is difficult to perform the exact integration and hence, impossible to solve the problem exactly. The use of encoder q(z|x) makes up for the deficiency by serving as a posterior approximation.

### Q2. What happens if the KL term is removed?

If KL divergence is excluded from the loss function, the model’s aim becomes the reconstruction of the input in the most precise manner. It means that there won’t be any need for the encoder to make sure the latent distributions are aligned with the prior distribution. This might result in the irregularity of the latent space and difficulty sampling from it.

### Q3. Why does the reparameterization trick work?

The reason why the reparametrization trick works is that it manages to separate the randomness from the variables of the model that can be learned through the training process. The random variable z is now expressed as z = μ + σε, which means that z depends on μ and σ differentiably. This trick prevents backpropagation from being interrupted by the sampling operation.

### Q4. What happens when μ = 0 and logvar = 0?

When both μ and logvar are zero, the distribution of the encoder becomes N(0,1). The prior is also N(0,1).Since, KL divergence calculates the difference between two distributions. Therefore, the value of KL divergence will be zero. This is consistent with the closed-form expression since all terms cancel each other within the summation
